In [1]:
import os
import sys
import re
import json
import random
import hashlib
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

print("=" * 80)
print("ENVIRONMENT & DEPENDENCY REPORT")
print("=" * 80)
print(f"Python Executable: {sys.executable}")
print(f"Python Version:    {sys.version}")
print(f"Pandas Version:    {pd.__version__}")
print(f"NumPy Version:     {np.__version__}")
print(f"PyArrow Version:   {pa.__version__}")
print("All core dependencies verified successfully.")

ENVIRONMENT & DEPENDENCY REPORT
Python Executable: /usr/bin/python3
Python Version:    3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Pandas Version:    2.3.3
NumPy Version:     2.1.3
PyArrow Version:   23.0.1
All core dependencies verified successfully.


In [2]:
# Master Configuration
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Kaggle vs Local Path Resolution
KAGGLE_INPUT_DIR = "/kaggle/input"
KAGGLE_WORKING_DIR = "/kaggle/working"

# Target output directory
OUTPUT_DIR = os.path.join(KAGGLE_WORKING_DIR, "processed_data") if os.path.exists(KAGGLE_WORKING_DIR) else "./processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dataset path candidate locators (Prioritizes exact user Kaggle dataset paths)
def resolve_path(candidates: List[str]) -> Optional[str]:
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

CSV_5K_PATH = resolve_path([
    # Exact user Kaggle upload path
    "/kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/medical_abstracts_rhetorical_roles.csv",
    "/kaggle/input/rhetorical-role-classification-in-medical-abstract/medical_abstracts_rhetorical_roles.csv",
    "/kaggle/input/medical-abstracts-rhetorical-roles/medical_abstracts_rhetorical_roles.csv",
    "/kaggle/input/medical_abstracts_rhetorical_roles.csv",
    "/kaggle/input/paper-collection/medical_abstracts_rhetorical_roles.csv",
    "medical_abstracts_rhetorical_roles.csv",
    "paper collection/medical_abstracts_rhetorical_roles.csv",
    os.path.join(os.getcwd(), "paper collection", "medical_abstracts_rhetorical_roles.csv"),
    os.path.join(os.getcwd(), "medical_abstracts_rhetorical_roles.csv")
])

CSV_META_PATH = resolve_path([
    "/kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/metadata.csv",
    "/kaggle/input/rhetorical-role-classification-in-medical-abstract/metadata.csv",
    "/kaggle/input/medical-abstracts-rhetorical-roles/metadata.csv",
    "/kaggle/input/metadata.csv",
    "/kaggle/input/paper-collection/metadata.csv",
    "metadata.csv",
    "paper collection/metadata.csv",
    os.path.join(os.getcwd(), "paper collection", "metadata.csv")
])

PUBMED_20K_DIR = resolve_path([
    # Exact user Kaggle upload path
    "/kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/PubMed_20k_RCT",
    "/kaggle/input/rhetorical-role-classification-in-medical-abstract/PubMed_20k_RCT",
    "/kaggle/input/pubmed-20k-rct/PubMed_20k_RCT",
    "/kaggle/input/pubmed-20k-rct",
    "/kaggle/input/pubmed_20k_rct",
    "/kaggle/input/paper-collection/PubMed_20k_RCT",
    "PubMed_20k_RCT",
    "paper collection/PubMed_20k_RCT",
    os.path.join(os.getcwd(), "paper collection", "PubMed_20k_RCT")
])

PUBMED_200K_DIR = resolve_path([
    # Exact user Kaggle upload path
    "/kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/PubMed_200k_RCT",
    "/kaggle/input/rhetorical-role-classification-in-medical-abstract/PubMed_200k_RCT",
    "/kaggle/input/pubmed-200k-rct/PubMed_200k_RCT",
    "/kaggle/input/pubmed-200k-rct",
    "/kaggle/input/pubmed_200k_rct",
    "/kaggle/input/paper-collection/PubMed_200k_RCT",
    "PubMed_200k_RCT",
    "paper collection/PubMed_200k_RCT",
    os.path.join(os.getcwd(), "paper collection", "PubMed_200k_RCT")
])

# Canonical Target Classes (5)
TARGET_CLASSES = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSION"]
LABEL2ID = {lbl: idx for idx, lbl in enumerate(TARGET_CLASSES)}
ID2LABEL = {idx: lbl for idx, lbl in enumerate(TARGET_CLASSES)}

# PubMed 200k processing limits (set None for full 2.2M sentences, or an integer for faster Kaggle experimentation)
PUBMED_200K_MAX_ABSTRACTS = None

print("Configuration initialized:")
print(f"  RANDOM_SEED:        {RANDOM_SEED}")
print(f"  OUTPUT_DIR:         {OUTPUT_DIR}")
print(f"  CSV_5K_PATH:        {CSV_5K_PATH}")
print(f"  CSV_META_PATH:      {CSV_META_PATH}")
print(f"  PUBMED_20K_DIR:     {PUBMED_20K_DIR}")
print(f"  PUBMED_200K_DIR:    {PUBMED_200K_DIR}")
print(f"  TARGET_CLASSES (5): {TARGET_CLASSES}")

Configuration initialized:
  RANDOM_SEED:        42
  OUTPUT_DIR:         /kaggle/working/processed_data
  CSV_5K_PATH:        /kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/medical_abstracts_rhetorical_roles.csv
  CSV_META_PATH:      /kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/metadata.csv
  PUBMED_20K_DIR:     /kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/PubMed_20k_RCT
  PUBMED_200K_DIR:    /kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/PubMed_200k_RCT
  TARGET_CLASSES (5): ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSION']


In [3]:
print("=" * 80)
print("FILESYSTEM AUTO-DISCOVERY & DIRECTORY AUDIT")
print("=" * 80)

def list_directory_tree(root_dir: str, max_depth: int = 2, current_depth: int = 0):
    if not os.path.exists(root_dir):
        print(f"Directory not found: {root_dir}")
        return
    prefix = "  " * current_depth
    print(f"{prefix}[DIR] {root_dir}")
    try:
        entries = sorted(os.listdir(root_dir))
    except Exception as e:
        print(f"{prefix}  <Error reading directory: {e}>")
        return
    for entry in entries:
        full_path = os.path.join(root_dir, entry)
        if os.path.isdir(full_path):
            if current_depth < max_depth:
                list_directory_tree(full_path, max_depth, current_depth + 1)
            else:
                print(f"{prefix}  [DIR] {entry} (contents hidden)")
        else:
            size_kb = os.path.getsize(full_path) / 1024
            print(f"{prefix}  - {entry} ({size_kb:,.1f} KB)")

if os.path.exists(KAGGLE_INPUT_DIR):
    print("Inspecting /kaggle/input:")
    list_directory_tree(KAGGLE_INPUT_DIR, max_depth=2)
else:
    print("/kaggle/input not present. Inspecting current working directory:")
    list_directory_tree(".", max_depth=2)

FILESYSTEM AUTO-DISCOVERY & DIRECTORY AUDIT
Inspecting /kaggle/input:
[DIR] /kaggle/input
  [DIR] /kaggle/input/datasets
    [DIR] /kaggle/input/datasets/siamabir
      [DIR] rhetorical-role-classification-in-medical-abstract (contents hidden)


In [4]:
print("=" * 80)
print("INSPECTING DATASET 1: medical_abstracts_rhetorical_roles.csv")
print("=" * 80)

if CSV_5K_PATH is None or not os.path.exists(CSV_5K_PATH):
    raise FileNotFoundError(f"Dataset 1 not found. Checked candidate paths: {CSV_5K_PATH}")

df_raw_5k = pd.read_csv(CSV_5K_PATH)
print(f"Shape: {df_raw_5k.shape[0]:,} rows, {df_raw_5k.shape[1]} columns")
print(f"Columns: {df_raw_5k.columns.tolist()}\n")

print("--- Null & Empty String Audit ---")
for col in df_raw_5k.columns:
    null_count = df_raw_5k[col].isnull().sum()
    empty_str_count = (df_raw_5k[col].astype(str).str.strip() == "").sum()
    print(f"  {col:15s}: Nulls = {null_count:5d} | Empty Strings = {empty_str_count:5d}")

duplicate_abstracts = df_raw_5k['Abstract'].duplicated().sum()
print(f"\nDuplicate Abstracts in Dataset 1: {duplicate_abstracts}")

# Check metadata for specialty
if CSV_META_PATH and os.path.exists(CSV_META_PATH):
    df_meta_5k = pd.read_csv(CSV_META_PATH)
    print(f"\nMetadata loaded: {len(df_meta_5k):,} records.")
    if 'Specialty' in df_meta_5k.columns:
        print("Specialty distribution (Top 5):")
        print(df_meta_5k['Specialty'].value_counts().head(5))
else:
    print("\nNote: metadata.csv not provided; specialty will default to 'Clinical Medicine' / None.")

INSPECTING DATASET 1: medical_abstracts_rhetorical_roles.csv
Shape: 5,000 rows, 6 columns
Columns: ['Abstract', 'Background', 'Objective', 'Methods', 'Results', 'Conclusion']

--- Null & Empty String Audit ---
  Abstract       : Nulls =     0 | Empty Strings =     0
  Background     : Nulls =     0 | Empty Strings =     0
  Objective      : Nulls =     0 | Empty Strings =     0
  Methods        : Nulls =     0 | Empty Strings =     0
  Results        : Nulls =     0 | Empty Strings =     0
  Conclusion     : Nulls =     0 | Empty Strings =     0

Duplicate Abstracts in Dataset 1: 0

Metadata loaded: 5,000 records.
Specialty distribution (Top 5):
Specialty
PLOS ONE (Clinical Medicine)      1740
PLOS ONE (Infectious Diseases)     985
PLOS ONE (Oncology)                966
PLOS ONE (Cardiology)              529
Clinical Medicine                  224
Name: count, dtype: int64


In [5]:
print("=" * 80)
print("INSPECTING DATASET 2: PubMed_20k_RCT")
print("=" * 80)

if PUBMED_20K_DIR is None or not os.path.exists(PUBMED_20K_DIR):
    raise FileNotFoundError(f"PubMed_20k_RCT directory not found: {PUBMED_20K_DIR}")

def audit_rct_directory(dir_path: str, dataset_name: str) -> Dict[str, Any]:
    stats = {}
    for fname in ["train.txt", "dev.txt", "test.txt"]:
        fpath = os.path.join(dir_path, fname)
        if not os.path.exists(fpath):
            print(f"Warning: Expected split file {fname} not found in {dir_path}")
            continue
        
        abstract_count = 0
        sentence_count = 0
        bad_tab_lines = 0
        raw_labels = {}
        
        with open(fpath, "r", encoding="utf-8") as f:
            for line in f:
                line_str = line.strip()
                if not line_str:
                    continue
                if line_str.startswith("###"):
                    abstract_count += 1
                else:
                    parts = line_str.split("	")
                    if len(parts) != 2:
                        bad_tab_lines += 1
                        continue
                    lbl, text = parts[0], parts[1]
                    sentence_count += 1
                    raw_labels[lbl] = raw_labels.get(lbl, 0) + 1
                    
        stats[fname] = {
            "file_size_bytes": os.path.getsize(fpath),
            "abstracts": abstract_count,
            "sentences": sentence_count,
            "bad_tab_lines": bad_tab_lines,
            "raw_labels": raw_labels
        }
        print(f"[{dataset_name}] {fname:9s}: {abstract_count:6,d} abstracts | {sentence_count:8,d} sentences | Bad tabs: {bad_tab_lines}")
        print(f"  Raw Labels: {raw_labels}")
    return stats

stats_20k = audit_rct_directory(PUBMED_20K_DIR, "PubMed 20k RCT")

INSPECTING DATASET 2: PubMed_20k_RCT
[PubMed 20k RCT] train.txt: 15,000 abstracts |  180,040 sentences | Bad tabs: 0
  Raw Labels: {'OBJECTIVE': 13839, 'METHODS': 59353, 'RESULTS': 57953, 'CONCLUSIONS': 27168, 'BACKGROUND': 21727}
[PubMed 20k RCT] dev.txt  :  2,500 abstracts |   30,212 sentences | Bad tabs: 0
  Raw Labels: {'BACKGROUND': 3449, 'OBJECTIVE': 2376, 'METHODS': 9964, 'RESULTS': 9841, 'CONCLUSIONS': 4582}
[PubMed 20k RCT] test.txt :  2,500 abstracts |   30,135 sentences | Bad tabs: 0
  Raw Labels: {'BACKGROUND': 3621, 'RESULTS': 9713, 'CONCLUSIONS': 4571, 'METHODS': 9897, 'OBJECTIVE': 2333}


In [6]:
print("=" * 80)
print("INSPECTING DATASET 3: PubMed_200k_RCT")
print("=" * 80)

if PUBMED_200K_DIR is None or not os.path.exists(PUBMED_200K_DIR):
    raise FileNotFoundError(f"PubMed_200k_RCT directory not found: {PUBMED_200K_DIR}")

# Check if train.7z needs unpacking
train_txt = os.path.join(PUBMED_200K_DIR, "train.txt")
train_7z = os.path.join(PUBMED_200K_DIR, "train.7z")
if not os.path.exists(train_txt) and os.path.exists(train_7z):
    print("train.txt not found but train.7z is present. Attempting decompression...")
    try:
        import py7zr
        with py7zr.SevenZipFile(train_7z, mode='r') as z:
            z.extractall(path=PUBMED_200K_DIR)
        print("Successfully extracted train.txt from train.7z!")
    except ImportError:
        print("py7zr not installed; using subprocess 7z command if available...")
        os.system(f'7z x "{train_7z}" -o"{PUBMED_200K_DIR}" -y')

stats_200k = audit_rct_directory(PUBMED_200K_DIR, "PubMed 200k RCT")

INSPECTING DATASET 3: PubMed_200k_RCT
[PubMed 200k RCT] train.txt: 190,654 abstracts | 2,211,861 sentences | Bad tabs: 0
  Raw Labels: {'BACKGROUND': 196689, 'METHODS': 722586, 'CONCLUSIONS': 339714, 'RESULTS': 766271, 'OBJECTIVE': 186601}
[PubMed 200k RCT] dev.txt  :  2,500 abstracts |   28,932 sentences | Bad tabs: 0
  Raw Labels: {'BACKGROUND': 2575, 'RESULTS': 9977, 'CONCLUSIONS': 4396, 'OBJECTIVE': 2425, 'METHODS': 9559}
[PubMed 200k RCT] test.txt :  2,500 abstracts |   29,493 sentences | Bad tabs: 0
  Raw Labels: {'BACKGROUND': 2663, 'METHODS': 9751, 'RESULTS': 10276, 'CONCLUSIONS': 4426, 'OBJECTIVE': 2377}


In [7]:
def split_sents(text: str) -> List[str]:
    """Splits a medical paragraph block into sentences while preserving medical abbreviations and decimal values."""
    if text is None or pd.isna(text):
        return []
    text = str(text).replace('\r', ' ').replace('\n', ' ').strip()
    if not text:
        return []
        
    abbrs = [
        'e.g.', 'i.e.', 'et al.', 'vs.', 'fig.', 'ref.', 'dr.', 'mr.', 'mrs.', 'ms.',
        'prof.', 'no.', 'vol.', 'pp.', 'approx.', 'cf.', 'min.', 'hr.', 'hrs.',
        'dept.', 'univ.', 'gen.', 'spec.', 'assn.', 'inc.', 'ltd.', 'corp.'
    ]
    # Protect abbreviations with temporary placeholder
    for abbr in abbrs:
        text = re.sub(r'\b' + re.escape(abbr), abbr.replace('.', '@#@'), text, flags=re.IGNORECASE)
    # Protect decimal numbers (e.g. 3.14, 0.05, p < 0.01)
    text = re.sub(r'(\d+)\.(\d+)', r'\1@#@\2', text)
    
    # Split on sentence boundaries followed by whitespace
    sents = re.split(r'(?<=[.!?])\s+', text)
    cleaned = [s.replace('@#@', '.').strip() for s in sents if s.strip()]
    return cleaned

def get_position_bucket(norm_pos: float) -> str:
    """Discretizes normalized position (0.0 to 1.0) into categorical buckets."""
    if norm_pos < 0.20:
        return "START"
    elif norm_pos < 0.40:
        return "EARLY"
    elif norm_pos < 0.70:
        return "MID"
    elif norm_pos < 0.90:
        return "LATE"
    else:
        return "END"

def map_label(raw_label: str) -> str:
    """Strictly maps dataset labels to the 5 canonical thesis classes."""
    raw_clean = str(raw_label).strip().upper()
    if raw_clean == "CONCLUSIONS":
        return "CONCLUSION"
    if raw_clean in TARGET_CLASSES:
        return raw_clean
    raise ValueError(f"Unknown label encountered outside the 5 target classes: '{raw_label}'")

# Verification of sentence splitting on sample medical string
sample_medical_text = "Severe malaria in children (e.g., Gabonese cohort) requires approx. 12.5 ml/kg fluid. We tested if TBW changes. Results: 19 moderate and 16 severe cases showed p < 0.05."
sample_split = split_sents(sample_medical_text)
print("Sentence Splitter Test:")
for idx, s in enumerate(sample_split, 1):
    print(f"  Sentence {idx}: {s}")
assert len(sample_split) == 3, f"Expected 3 sentences, got {len(sample_split)}"
print("Sentence splitter verified successfully!")

Sentence Splitter Test:
  Sentence 1: Severe malaria in children (e.g., Gabonese cohort) requires approx. 12.5 ml/kg fluid.
  Sentence 2: We tested if TBW changes.
  Sentence 3: Results: 19 moderate and 16 severe cases showed p < 0.05.
Sentence splitter verified successfully!


In [8]:
print("=" * 80)
print("PROCESSING DATASET 1: medical_abstracts_rhetorical_roles.csv")
print("=" * 80)

# 1. Deterministic paper-level partitioning
num_papers = len(df_raw_5k)
paper_ids = list(range(1, num_papers + 1))
random.seed(RANDOM_SEED)
random.shuffle(paper_ids)

train_split_count = int(num_papers * 0.70)  # 3500
val_split_count = int(num_papers * 0.15)    # 750
test_split_count = num_papers - train_split_count - val_split_count # 750

train_paper_ids = set(paper_ids[:train_split_count])
val_paper_ids = set(paper_ids[train_split_count:train_split_count + val_split_count])
test_paper_ids = set(paper_ids[train_split_count + val_split_count:])

print(f"Paper Split Summary: Train={len(train_paper_ids)}, Val={len(val_paper_ids)}, Test={len(test_paper_ids)} papers.")

# 2. Specialty lookup map
specialty_map = {}
if CSV_META_PATH and os.path.exists(CSV_META_PATH):
    df_meta_5k = pd.read_csv(CSV_META_PATH)
    for idx, row in df_meta_5k.iterrows():
        pid = row.get("ID", idx + 1)
        spec = row.get("Specialty", row.get("Journal", "Clinical Medicine"))
        specialty_map[pid] = spec

records_5k = []
for idx, row in df_raw_5k.iterrows():
    pid = idx + 1
    split_name = "train" if pid in train_paper_ids else ("val" if pid in val_paper_ids else "test")
    specialty = specialty_map.get(pid, "Clinical Medicine")
    
    sections = [
        ("BACKGROUND", split_sents(row["Background"])),
        ("OBJECTIVE", split_sents(row["Objective"])),
        ("METHODS", split_sents(row["Methods"])),
        ("RESULTS", split_sents(row["Results"])),
        ("CONCLUSION", split_sents(row["Conclusion"]))
    ]
    
    all_sents = []
    for label, s_list in sections:
        for s in s_list:
            all_sents.append((s, label))
            
    total_l = len(all_sents)
    if total_l == 0:
        continue
        
    for line_no, (stext, lbl) in enumerate(all_sents):
        prev_s = all_sents[line_no - 1][0] if line_no > 0 else "[START_OF_ABSTRACT]"
        next_s = all_sents[line_no + 1][0] if line_no < total_l - 1 else "[END_OF_ABSTRACT]"
        norm_pos = round(line_no / total_l, 4)
        pos_bucket = get_position_bucket(norm_pos)
        
        records_5k.append({
            "dataset_name": "medical_abstracts_5k",
            "paper_id": str(pid),
            "line_number": line_no,
            "total_lines": total_l,
            "normalized_pos": norm_pos,
            "position_bucket": pos_bucket,
            "prev_sentence": prev_s,
            "sentence_text": stext,
            "next_sentence": next_s,
            "specialty": specialty,
            "label": map_label(lbl),
            "split": split_name
        })

df_5k = pd.DataFrame(records_5k)
print(f"Dataset 1 processed: {len(df_5k):,} sentences across {df_5k['paper_id'].nunique():,} abstracts.")
print("Split Distribution:")
print(df_5k['split'].value_counts())
print("\nClass Distribution:")
print(df_5k['label'].value_counts())

PROCESSING DATASET 1: medical_abstracts_rhetorical_roles.csv
Paper Split Summary: Train=3500, Val=750, Test=750 papers.
Dataset 1 processed: 64,877 sentences across 5,000 abstracts.
Split Distribution:
split
train    45302
test      9845
val       9730
Name: count, dtype: int64

Class Distribution:
label
RESULTS       20623
METHODS       16664
BACKGROUND    11056
CONCLUSION     9871
OBJECTIVE      6663
Name: count, dtype: int64


In [9]:
print("=" * 80)
print("PROCESSING DATASET 2: PubMed_20k_RCT")
print("=" * 80)

def parse_rct_dataset(dir_path: str, dataset_name: str, max_abstracts: Optional[int] = None) -> pd.DataFrame:
    records = []
    split_file_map = {
        "train": "train.txt",
        "val": "dev.txt",
        "test": "test.txt"
    }
    
    for split_name, fname in split_file_map.items():
        fpath = os.path.join(dir_path, fname)
        if not os.path.exists(fpath):
            continue
            
        current_pid = None
        current_abstract_lines = []
        abstract_count_split = 0
        
        def flush_abstract(pid: str, lines: List[Tuple[str, str]], s_name: str):
            tot_l = len(lines)
            if tot_l == 0:
                return
            for idx_line, (stext, rlabel) in enumerate(lines):
                norm_p = round(idx_line / tot_l, 4)
                p_bucket = get_position_bucket(norm_p)
                p_sent = lines[idx_line - 1][0] if idx_line > 0 else "[START_OF_ABSTRACT]"
                n_sent = lines[idx_line + 1][0] if idx_line < tot_l - 1 else "[END_OF_ABSTRACT]"
                
                records.append({
                    "dataset_name": dataset_name,
                    "paper_id": str(pid),
                    "line_number": idx_line,
                    "total_lines": tot_l,
                    "normalized_pos": norm_p,
                    "position_bucket": p_bucket,
                    "prev_sentence": p_sent,
                    "sentence_text": stext,
                    "next_sentence": n_sent,
                    "specialty": None,  # Real data lacks specialty metadata; do NOT fabricate
                    "label": map_label(rlabel),
                    "split": s_name
                })

        with open(fpath, "r", encoding="utf-8") as f:
            for line in f:
                line_str = line.strip()
                if not line_str:
                    if current_pid is not None and current_abstract_lines:
                        flush_abstract(current_pid, current_abstract_lines, split_name)
                        abstract_count_split += 1
                        current_pid = None
                        current_abstract_lines = []
                        if max_abstracts and abstract_count_split >= max_abstracts:
                            break
                    continue
                if line_str.startswith("###"):
                    if current_pid is not None and current_abstract_lines:
                        flush_abstract(current_pid, current_abstract_lines, split_name)
                        abstract_count_split += 1
                        current_abstract_lines = []
                        if max_abstracts and abstract_count_split >= max_abstracts:
                            break
                    current_pid = line_str.replace("###", "").strip()
                else:
                    parts = line_str.split("	")
                    if len(parts) == 2:
                        current_abstract_lines.append((parts[1].strip(), parts[0].strip()))
                        
            # Flush final abstract if file ends without blank line
            if current_pid is not None and current_abstract_lines:
                flush_abstract(current_pid, current_abstract_lines, split_name)
                
    return pd.DataFrame(records)

df_20k = parse_rct_dataset(PUBMED_20K_DIR, "pubmed_20k_rct")
print(f"Dataset 2 processed: {len(df_20k):,} sentences across {df_20k['paper_id'].nunique():,} abstracts.")
print("Split Distribution:")
print(df_20k['split'].value_counts())
print("\nClass Distribution:")
print(df_20k['label'].value_counts())

PROCESSING DATASET 2: PubMed_20k_RCT
Dataset 2 processed: 240,387 sentences across 20,000 abstracts.
Split Distribution:
split
train    180040
val       30212
test      30135
Name: count, dtype: int64

Class Distribution:
label
METHODS       79214
RESULTS       77507
CONCLUSION    36321
BACKGROUND    28797
OBJECTIVE     18548
Name: count, dtype: int64


In [10]:
print("=" * 80)
print("PROCESSING DATASET 3: PubMed_200k_RCT")
print("=" * 80)

df_200k = parse_rct_dataset(PUBMED_200K_DIR, "pubmed_200k_rct", max_abstracts=PUBMED_200K_MAX_ABSTRACTS)
print(f"Dataset 3 processed: {len(df_200k):,} sentences across {df_200k['paper_id'].nunique():,} abstracts.")
print("Split Distribution:")
print(df_200k['split'].value_counts())
print("\nClass Distribution:")
print(df_200k['label'].value_counts())

PROCESSING DATASET 3: PubMed_200k_RCT
Dataset 3 processed: 2,270,286 sentences across 195,654 abstracts.
Split Distribution:
split
train    2211861
test       29493
val        28932
Name: count, dtype: int64

Class Distribution:
label
RESULTS       786524
METHODS       741896
CONCLUSION    348536
BACKGROUND    201927
OBJECTIVE     191403
Name: count, dtype: int64


In [11]:
print("=" * 80)
print("ZERO-LEAKAGE AUDIT & DATA QUALITY ASSERTS")
print("=" * 80)

datasets_to_check = [
    ("medical_abstracts_5k", df_5k),
    ("pubmed_20k_rct", df_20k),
    ("pubmed_200k_rct", df_200k)
]

for dname, df_cur in datasets_to_check:
    print(f"\n--- Auditing {dname} ---")
    pids_train = set(df_cur[df_cur['split'] == 'train']['paper_id'])
    pids_val = set(df_cur[df_cur['split'] == 'val']['paper_id'])
    pids_test = set(df_cur[df_cur['split'] == 'test']['paper_id'])
    
    # 1. Leakage assertion
    leak_tv = pids_train.intersection(pids_val)
    leak_tt = pids_train.intersection(pids_test)
    leak_vt = pids_val.intersection(pids_test)
    
    assert len(leak_tv) == 0, f"DATA LEAKAGE DETECTED in {dname}: {len(leak_tv)} paper_ids overlap between train & val!"
    assert len(leak_tt) == 0, f"DATA LEAKAGE DETECTED in {dname}: {len(leak_tt)} paper_ids overlap between train & test!"
    assert len(leak_vt) == 0, f"DATA LEAKAGE DETECTED in {dname}: {len(leak_vt)} paper_ids overlap between val & test!"
    print(f"  [PASS] Zero paper-level data leakage: Train({len(pids_train)}), Val({len(pids_val)}), Test({len(pids_test)}) are 100% disjoint.")
    
    # 2. Duplicate sentence checks
    s_train = set(df_cur[df_cur['split'] == 'train']['sentence_text'])
    s_val = set(df_cur[df_cur['split'] == 'val']['sentence_text'])
    s_test = set(df_cur[df_cur['split'] == 'test']['sentence_text'])
    
    dup_tv = len(s_train.intersection(s_val))
    dup_tt = len(s_train.intersection(s_test))
    dup_vt = len(s_val.intersection(s_test))
    print(f"  Cross-Split Duplicate Sentences (common academic boilerplate phrases):")
    print(f"    Train & Val: {dup_tv} | Train & Test: {dup_tt} | Val & Test: {dup_vt}")
    
    # 3. Sentences per abstract stats
    s_per_abs = df_cur.groupby('paper_id').size()
    print(f"  Sentences per Abstract:")
    print(f"    Count: {s_per_abs.count():,} abstracts")
    print(f"    Mean:  {s_per_abs.mean():.2f} +/- {s_per_abs.std():.2f}")
    print(f"    Range: Min = {s_per_abs.min()}, Median = {s_per_abs.median():.0f}, Max = {s_per_abs.max()}")

print("\n" + "=" * 80)
print("RANDOM EXAMPLES PER RHETORICAL ROLE (5 Target Classes):")
print("=" * 80)
for role in TARGET_CLASSES:
    sample = df_5k[df_5k['label'] == role].sample(1, random_state=RANDOM_SEED).iloc[0]
    print(f"\n--- Role: [{role}] ---")
    print(f"  Abstract ID:    {sample['paper_id']} (Line {sample['line_number'] + 1} of {sample['total_lines']}, Bucket: {sample['position_bucket']})")
    print(f"  Prev Sentence:  {sample['prev_sentence'][:90]}...")
    print(f"  TARGET SENTENCE: {sample['sentence_text']}")
    print(f"  Next Sentence:  {sample['next_sentence'][:90]}...")

ZERO-LEAKAGE AUDIT & DATA QUALITY ASSERTS

--- Auditing medical_abstracts_5k ---
  [PASS] Zero paper-level data leakage: Train(3500), Val(750), Test(750) are 100% disjoint.
  Cross-Split Duplicate Sentences (common academic boilerplate phrases):
    Train & Val: 17 | Train & Test: 13 | Val & Test: 8
  Sentences per Abstract:
    Count: 5,000 abstracts
    Mean:  12.98 +/- 4.81
    Range: Min = 5, Median = 12, Max = 71

--- Auditing pubmed_20k_rct ---
  [PASS] Zero paper-level data leakage: Train(15000), Val(2500), Test(2500) are 100% disjoint.
  Cross-Split Duplicate Sentences (common academic boilerplate phrases):
    Train & Val: 206 | Train & Test: 206 | Val & Test: 94
  Sentences per Abstract:
    Count: 20,000 abstracts
    Mean:  12.02 +/- 3.07
    Range: Min = 4, Median = 12, Max = 31

--- Auditing pubmed_200k_rct ---
  [PASS] Zero paper-level data leakage: Train(190654), Val(2500), Test(2500) are 100% disjoint.
  Cross-Split Duplicate Sentences (common academic boilerplate phra

In [12]:
print("=" * 80)
print("EXPORTING MULTI-FORMAT DATASET ARTIFACTS")
print("=" * 80)

def compute_sha256(filepath: str) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

manifest = {}

# 1. Save Parquet and CSV for Dataset 1
path_5k_pq = os.path.join(OUTPUT_DIR, "medical_abstracts_5k_sentences.parquet")
path_5k_csv = os.path.join(OUTPUT_DIR, "medical_abstracts_5k_sentences.csv")
df_5k.to_parquet(path_5k_pq, index=False)
df_5k.to_csv(path_5k_csv, index=False, encoding="utf-8-sig")

manifest["medical_abstracts_5k_sentences.parquet"] = {
    "rows": len(df_5k),
    "size_bytes": os.path.getsize(path_5k_pq),
    "sha256": compute_sha256(path_5k_pq)
}
manifest["medical_abstracts_5k_sentences.csv"] = {
    "rows": len(df_5k),
    "size_bytes": os.path.getsize(path_5k_csv),
    "sha256": compute_sha256(path_5k_csv)
}
print(f"Exported: medical_abstracts_5k ({len(df_5k):,} rows) -> Parquet & CSV")

# 2. Save Parquet and CSV for Dataset 2 (PubMed 20k)
path_20k_pq = os.path.join(OUTPUT_DIR, "pubmed_20k_sentences.parquet")
path_20k_csv = os.path.join(OUTPUT_DIR, "pubmed_20k_sentences.csv")
df_20k.to_parquet(path_20k_pq, index=False)
df_20k.to_csv(path_20k_csv, index=False, encoding="utf-8-sig")

manifest["pubmed_20k_sentences.parquet"] = {
    "rows": len(df_20k),
    "size_bytes": os.path.getsize(path_20k_pq),
    "sha256": compute_sha256(path_20k_pq)
}
manifest["pubmed_20k_sentences.csv"] = {
    "rows": len(df_20k),
    "size_bytes": os.path.getsize(path_20k_csv),
    "sha256": compute_sha256(path_20k_csv)
}
print(f"Exported: pubmed_20k_rct ({len(df_20k):,} rows) -> Parquet & CSV")

# 3. Save Parquet for Dataset 3 (PubMed 200k)
path_200k_pq = os.path.join(OUTPUT_DIR, "pubmed_200k_sentences.parquet")
df_200k.to_parquet(path_200k_pq, index=False)
manifest["pubmed_200k_sentences.parquet"] = {
    "rows": len(df_200k),
    "size_bytes": os.path.getsize(path_200k_pq),
    "sha256": compute_sha256(path_200k_pq)
}
print(f"Exported: pubmed_200k_rct ({len(df_200k):,} rows) -> Parquet")

# 4. Save Combined Parquet (5k + 20k)
df_combined = pd.concat([df_5k, df_20k], ignore_index=True)
path_comb_pq = os.path.join(OUTPUT_DIR, "combined_sentences.parquet")
df_combined.to_parquet(path_comb_pq, index=False)
manifest["combined_sentences.parquet"] = {
    "rows": len(df_combined),
    "size_bytes": os.path.getsize(path_comb_pq),
    "sha256": compute_sha256(path_comb_pq)
}
print(f"Exported: combined_sentences ({len(df_combined):,} rows) -> Parquet")

# 5. Export ChatML JSONL for LLM Instruction Fine-Tuning
def export_chatml_jsonl(df_source: pd.DataFrame, out_fname: str):
    out_path = os.path.join(OUTPUT_DIR, out_fname)
    count = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in df_source.iterrows():
            specialty_clause = f"Medical Specialty: {row['specialty']}\n" if row['specialty'] else ""
            user_content = (
                f"You are an expert biomedical NLP system. Classify the rhetorical role of the target sentence within the medical research abstract.\n"
                f"{specialty_clause}"
                f"Sentence Position: Line {row['line_number'] + 1} of {row['total_lines']} ({row['position_bucket']} position, relative {row['normalized_pos']:.2f})\n"
                f"Preceding Sentence: {row['prev_sentence']}\n"
                f"Target Sentence: \"{row['sentence_text']}\"\n"
                f"Succeeding Sentence: {row['next_sentence']}\n\n"
                f"Select exactly one rhetorical role from: [BACKGROUND, OBJECTIVE, METHODS, RESULTS, CONCLUSION]."
            )
            item = {
                "dataset_name": row["dataset_name"],
                "paper_id": row["paper_id"],
                "split": row["split"],
                "messages": [
                    {
                        "role": "system",
                        "content": "You are a specialized biomedical AI classifier trained to categorize sentences in medical research abstracts into their standard rhetorical discourse roles (BACKGROUND, OBJECTIVE, METHODS, RESULTS, CONCLUSION)."
                    },
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": row["label"]}
                ]
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            count += 1
            
    manifest[out_fname] = {
        "rows": count,
        "size_bytes": os.path.getsize(out_path),
        "sha256": compute_sha256(out_path)
    }
    print(f"Exported: {out_fname} ({count:,} records)")

export_chatml_jsonl(df_5k, "llm_instruction_5k.jsonl")
export_chatml_jsonl(df_20k, "llm_instruction_pubmed20k.jsonl")

# 6. Save Dataset Metadata JSON
meta_payload = {
    "target_classes": TARGET_CLASSES,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "datasets": {
        "medical_abstracts_5k": {
            "total_sentences": len(df_5k),
            "abstracts": int(df_5k["paper_id"].nunique()),
            "splits": df_5k["split"].value_counts().to_dict(),
            "classes": df_5k["label"].value_counts().to_dict()
        },
        "pubmed_20k_rct": {
            "total_sentences": len(df_20k),
            "abstracts": int(df_20k["paper_id"].nunique()),
            "splits": df_20k["split"].value_counts().to_dict(),
            "classes": df_20k["label"].value_counts().to_dict()
        },
        "pubmed_200k_rct": {
            "total_sentences": len(df_200k),
            "abstracts": int(df_200k["paper_id"].nunique()),
            "splits": df_200k["split"].value_counts().to_dict(),
            "classes": df_200k["label"].value_counts().to_dict()
        }
    }
}
meta_path = os.path.join(OUTPUT_DIR, "dataset_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_payload, f, indent=2)
manifest["dataset_metadata.json"] = {
    "rows": 1,
    "size_bytes": os.path.getsize(meta_path),
    "sha256": compute_sha256(meta_path)
}

# 7. Save Manifest JSON
manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 80)
print("FINAL DATASET MANIFEST (Stored in /kaggle/working/processed_data/manifest.json):")
print("=" * 80)
for fname, meta in manifest.items():
    print(f"  {fname:42s}: {meta['rows']:8,d} rows | {meta['size_bytes'] / (1024*1024):6.2f} MB | SHA256: {meta['sha256'][:12]}...")

EXPORTING MULTI-FORMAT DATASET ARTIFACTS
Exported: medical_abstracts_5k (64,877 rows) -> Parquet & CSV
Exported: pubmed_20k_rct (240,387 rows) -> Parquet & CSV
Exported: pubmed_200k_rct (2,270,286 rows) -> Parquet
Exported: combined_sentences (305,264 rows) -> Parquet
Exported: llm_instruction_5k.jsonl (64,877 records)
Exported: llm_instruction_pubmed20k.jsonl (240,387 records)

FINAL DATASET MANIFEST (Stored in /kaggle/working/processed_data/manifest.json):
  medical_abstracts_5k_sentences.parquet    :   64,877 rows |  14.20 MB | SHA256: b21519cf80cc...
  medical_abstracts_5k_sentences.csv        :   64,877 rows |  32.20 MB | SHA256: 69d920a21e53...
  pubmed_20k_sentences.parquet              :  240,387 rows |  54.55 MB | SHA256: d19d40cee600...
  pubmed_20k_sentences.csv                  :  240,387 rows | 113.61 MB | SHA256: f3b78bd90659...
  pubmed_200k_sentences.parquet             : 2,270,286 rows | 505.58 MB | SHA256: a70121af7b03...
  combined_sentences.parquet                : 